In [ ]:
session# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [8]:
PATH = "/kaggle/input/competitions/ieee-fraud-detection"
id = pd.read_csv(PATH+"/train_identity.csv")
tr = pd.read_csv(PATH+"/train_transaction.csv")

In [9]:
!pip install mlflow dagshub -q

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import mlflow.sklearn
import mlflow
import dagshub

In [11]:
TARGET = "isFraud"
TR_MISS_TH = 0.8
ID_MISS_TH = 0.7
CARDINALITY_TH = 15
CORRELATION_TH = 0.8

In [12]:
dagshub.init(repo_owner='lukaLomadze', repo_name='ML_Assignment_2', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow")
mlflow.set_experiment('AdaBoost')

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7fd0985b-d05c-483e-bd21-a29d71a37c92&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=752b5a6f40ae14ce415718e755ad0fabf29fdcfd976303c80df726ff6fa2ff07




Output()

Accessing as lukaLomadze

Initialized MLflow to track repo "lukaLomadze/ML_Assignment_2"

Repository lukaLomadze/ML_Assignment_2 initialized!

2026/05/04 15:20:12 INFO mlflow.tracking.fluent: Experiment with name 'AdaBoost' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f835c13477aa43b5bd51ee32d1f25677', creation_time=1777908012711, experiment_id='4', last_update_time=1777908012711, lifecycle_stage='active', name='AdaBoost', tags={}, trace_location=None, workspace='default'>

# Cleaning

In [13]:
class DropHighMissing(BaseEstimator, TransformerMixin):
    def __init__(self, tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=None):
        self.tr_th = tr_th
        self.id_th = id_th
        self.identity_cols = identity_cols

    def fit(self, X, y=None):
        id_set = set(self.identity_cols) if self.identity_cols else set()
        keep = []
        for col in X.columns:
            miss_rate = X[col].isnull().mean()
            threshold = self.id_th if col in id_set else self.tr_th
            if miss_rate <= threshold:
                keep.append(col)
        self.cols_ = keep
        return self

    def transform(self, X):
        return X[self.cols_]

In [ ]:
train = tr.merge(id, on="TransactionID", how="left")
y = train[TARGET].copy()

print(f"Train shape: {train.shape}")
print(f"Fraud rate: {y.mean()}")

Train shape: (590540, 434)
Fraud rate: 0.0350


In [15]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(exclude='object').columns.tolist()

In [16]:
with mlflow.start_run(run_name="adaboost_cleaning"):
    mlflow.log_param("TR_MISS_TH", TR_MISS_TH)
    mlflow.log_param("ID_MISS_TH", ID_MISS_TH)
    mlflow.log_param("Final shape", f"{train.shape[0]} X {train.shape[1]}")
    mlflow.log_param("Num cols", len(num_cols))
    mlflow.log_param("Cat cols", len(cat_cols))
    mlflow.log_param("Fraud rate", float(y.mean()))

🏃 View run adaboost_cleaning at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/4/runs/bb383165cc2c4f07b29a766abd15008a
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/4


# Feature Engineering

In [17]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, identity_cols=None, identity_ids=None, target=None):
        self.identity_cols = identity_cols
        self.identity_ids = identity_ids
        self.target = target

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if "TransactionAmt" in X:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
            X["TransactionAmt_sqrt"] = np.sqrt(X["TransactionAmt"])
        if "TransactionDT" in X:
            X["TransactionHour"] = (X["TransactionDT"] // 3600) % 24
            X["TransactionDay"] = (X["TransactionDT"] // 86400) % 7
            X["TransactionWeek"] = (X["TransactionDT"] // 86400) // 7

        if self.identity_ids is not None and "TransactionID" in X.columns:
            X["has_identity"] = X["TransactionID"].isin(self.identity_ids).astype(int)

        drop_cols = []
        if "TransactionID" in X.columns:
            drop_cols.append("TransactionID")
        if self.target and self.target in X.columns:
            drop_cols.append(self.target)
        if drop_cols:
            X = X.drop(columns=drop_cols)

        return X

In [18]:
low_card_cols = [c for c in cat_cols if train[c].nunique() <= CARDINALITY_TH]
high_card_cols = [c for c in cat_cols if train[c].nunique() > CARDINALITY_TH]

print(f"Low cardinality cols: {len(low_card_cols)}")
print(f"High cardinality cols: {len(high_card_cols)}")

Low cardinality cols: 25
High cardinality cols: 6


In [19]:
with mlflow.start_run(run_name="adaboost_feature_engineering"):
    mlflow.log_param("New features", "TransactionDay, TransactionHour, TransactionWeek, TransactionAmt_log, TransactionAmt_sqrt, has_identity")
    mlflow.log_param("CARDINALITY_TH", CARDINALITY_TH)
    mlflow.log_param("Low cardinality cols", len(low_card_cols))
    mlflow.log_param("High cardinality cols", len(high_card_cols))

🏃 View run adaboost_feature_engineering at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/4/runs/d7f8513acba04a0ea5d0abc3851d4b9f
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/4


# Feature Selection 


# Preprocessor 

In [20]:
class Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, cardinality_th=15):
        self.cardinality_th = cardinality_th

    def _build(self, X):
        num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        cat_cols = X.select_dtypes(include="object").columns.tolist()

        low_card = [c for c in cat_cols if X[c].nunique() <= self.cardinality_th]
        high_card = [c for c in cat_cols if X[c].nunique() > self.cardinality_th]

        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        ohe_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        ord_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ])

        transformers = [("num", num_pipe, num_cols)]
        if low_card:
            transformers.append(("ohe", ohe_pipe, low_card))
        if high_card:
            transformers.append(("ord", ord_pipe, high_card))

        return ColumnTransformer(transformers, remainder="drop")

    def fit(self, X, y=None):
        self.ct_ = self._build(X)
        self.ct_.fit(X, y)
        return self

    def transform(self, X):
        return self.ct_.transform(X)

# Build Pipeline

In [26]:
def build_pipeline(n_estimators, learning_rate,  sampling, max_depth=1):
    sampler = SMOTE(random_state=42) if sampling == "smote" else RandomUnderSampler(random_state=42)

    identity_cols = [c for c in id.columns if c != "TransactionID"]
    identity_ids = set(id["TransactionID"].unique())

    base_estimator = DecisionTreeClassifier(max_depth=max_depth, random_state=42)

    return ImbPipeline([
        ("drop", DropHighMissing(tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=identity_cols)),
        ("feat", FeatureEngineer(identity_cols=identity_cols, identity_ids=identity_ids, target=TARGET)),
        
        ("prep", Preprocessor(cardinality_th=CARDINALITY_TH)),
        ("sample", sampler),
        ("model", AdaBoostClassifier(
            estimator=base_estimator,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
        
            random_state=42
        ))
    ])

# Training

In [30]:
PARAM_GRID = [
    dict(n_estimators=50, learning_rate=1.0, sampling="smote", max_depth=1),
    dict(n_estimators=100, learning_rate=1.0, sampling="smote", max_depth=1),
    dict(n_estimators=200, learning_rate=1.0, sampling="smote", max_depth=1),
    dict(n_estimators=100, learning_rate=0.5, sampling="smote", max_depth=1),
    dict(n_estimators=100, learning_rate=1.5, sampling="smote", max_depth=1),
    dict(n_estimators=100, learning_rate=1.0, sampling="smote", max_depth=1),
    dict(n_estimators=100, learning_rate=1.0, sampling="undersample", max_depth=1),
    dict(n_estimators=100, learning_rate=1.0, sampling="smote", max_depth=2),
    dict(n_estimators=100, learning_rate=1.0, sampling="smote", max_depth=3),
]

CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_auc = -1
best_params = None
results = []

X = train

In [ ]:
for p in PARAM_GRID:
    with mlflow.start_run(run_name=f"ADA_n={p['n_estimators']}_lr={p['learning_rate']}_sample={p['sampling']}_depth={p['max_depth']}"):
        pipe = build_pipeline(p["n_estimators"], p["learning_rate"], p["sampling"], p["max_depth"])
        scores = cross_val_score(pipe, X, y, cv=CV, scoring="roc_auc", n_jobs=1)
        mean_auc = scores.mean()
        std_auc = scores.std()

        mlflow.log_params(p)
        mlflow.log_metric("cv_auc_mean", mean_auc)
        mlflow.log_metric("cv_auc_std", std_auc)

        results.append((p, mean_auc))

        if mean_auc > best_auc:
            best_auc = mean_auc
            best_params = p
            
        print(f"Params: {p} | AUC: {mean_auc} (+/- {std_auc})")

In [ ]:
print(f"\nBest params: {best_params}  |  CV AUC: {best_auc}")

best_pipe = build_pipeline(
    n_estimators=best_params["n_estimators"],
    learning_rate=best_params["learning_rate"],
    algorithm=best_params["algorithm"],
    sampling=best_params["sampling"],
    max_depth=best_params["max_depth"]
)

with mlflow.start_run(run_name="ADA_best_registered") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("cv_auc_mean", best_auc)

    best_pipe.fit(X, y)

    model_info = mlflow.sklearn.log_model(
        sk_model=best_pipe,
        artifact_path="model",
        registered_model_name="ADA_Best",
    )

    print(f"Model registered: {model_info.model_uri}")
    print(f"Run ID: {run.info.run_id}")